<a href="https://colab.research.google.com/github/Dracarys38/Machyne-navchanya/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0_%D1%80%D0%BE%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9612_%D0%9C%D0%9D%2C_%D0%9F%D0%BE%D1%81%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D1%8F%D0%BA_%D0%86%D0%B3%D0%BE%D1%80_%D0%A1%D0%B5%D1%80%D0%B3%D1%96%D0%B9%D0%BE%D0%B2%D0%B8%D1%87_%D0%A4%D0%86%D0%A2_3_15%2C_10_%D0%B2%D0%B0%D1%80%D1%96%D0%B0%D0%BD%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1.1 Імпорт бібліотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.19.0
GPU available: False


### 1.2 Завантаження та підготовка даних

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Нормалізація та зміна форми для CNN: (N, 28, 28, 1)
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(f"Тренувальна вибірка: {X_train.shape} | Мітки: {y_train.shape}")
print(f"Тестова вибірка:     {X_test.shape}  | Мітки: {y_test.shape}")

Тренувальна вибірка: (60000, 28, 28, 1) | Мітки: (60000,)
Тестова вибірка:     (10000, 28, 28, 1)  | Мітки: (10000,)


### 1.3 Побудова архітектури CNN

In [ ]:
def build_cnn_fashion(filters_1=32, filters_2=64, filters_3=128,
                      dense_units=256, dropout=0.4, lr=0.001):
    """Побудова CNN для Fashion MNIST."""
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),

        # Блок 1
        layers.Conv2D(filters_1, (3, 3), padding='same', activation='relu', name='conv1_1'),
        layers.BatchNormalization(name='bn1_1'),
        layers.Conv2D(filters_1, (3, 3), padding='same', activation='relu', name='conv1_2'),
        layers.BatchNormalization(name='bn1_2'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        layers.Dropout(dropout / 2, name='drop1'),

        # Блок 2
        layers.Conv2D(filters_2, (3, 3), padding='same', activation='relu', name='conv2_1'),
        layers.BatchNormalization(name='bn2_1'),
        layers.Conv2D(filters_2, (3, 3), padding='same', activation='relu', name='conv2_2'),
        layers.BatchNormalization(name='bn2_2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        layers.Dropout(dropout / 2, name='drop2'),

        # Блок 3
        layers.Conv2D(filters_3, (3, 3), padding='same', activation='relu', name='conv3_1'),
        layers.BatchNormalization(name='bn3_1'),
        layers.GlobalAveragePooling2D(name='gap'),

        # Класифікатор
        layers.Dense(dense_units, activation='relu', name='dense_1'),
        layers.Dropout(dropout, name='drop3'),
        layers.Dense(10, activation='softmax', name='output')
    ], name='FashionCNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_cnn_fashion()
cnn_model.summary()

Model: "FashionCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1_1 (Conv2D)                │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_1 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_1 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_1 (BatchNormalization)      │ (None, 7, 7, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,722 (686.41 KB)

 Trainable params: 175,082 (683.91 KB)

 Non-trainable params: 640 (2.50 KB)

### 1.4 Аугментація даних та навчання

In [ ]:
# ── Прискорення тренування ──────────────────────────────────────────────
# 1. Mixed precision (FP16) — ~2x швидше на GPU/TPU
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# 2. Аугментація прямо в моделі (виконується на GPU, а не CPU)
aug_layer = keras.Sequential([
    layers.RandomFlip('horizontal'),   # найлегша операція
    layers.RandomRotation(0.05),       # малий кут — швидко
], name='augmentation')

# Перебудувати модель з аугментацією всередині та FP16-сумісним output
def build_cnn_fast(filters_1=32, filters_2=64, filters_3=128,
                   dense_units=128, dropout=0.3, lr=0.001):
    inputs = keras.Input(shape=(28, 28, 1))
    x = aug_layer(inputs)                          # аугментація на GPU

    x = layers.Conv2D(filters_1, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout / 2)(x)

    x = layers.Conv2D(filters_2, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout / 2)(x)

    x = layers.Conv2D(filters_3, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    # dtype='float32' для стабільності FP16 у softmax
    outputs = layers.Dense(10, activation='softmax', dtype='float32', name='output')(x)

    model = keras.Model(inputs, outputs, name='FashionCNN_Fast')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_cnn_fast()
cnn_model.summary()

# 3. Колбеки: менший patience — раніше зупиняється
callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

# 4. tf.data pipeline: аугментація вже в моделі, тут тільки shuffle+batch
BATCH_SIZE = 256   # більший batch → менше ітерацій за епоху
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = (tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(10000, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=True)   # drop_remainder → XLA-компіляція
    .prefetch(AUTOTUNE)
)

val_ds = (tf.data.Dataset
    .from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# 5. Навчання — max 30 епох (EarlyStopping зупинить раніше)
history_cnn = cnn_model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    callbacks=callbacks_cnn,
    verbose=1
)

Model: "FashionCNN_Fast"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,370 (435.04 KB)

 Trainable params: 110,922 (433.29 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
  6/234 ━━━━━━━━━━━━━━━━━━━━ 1:20:33 21s/step - accuracy: 0.2002 - loss: 2.1437

### 1.5 Криві навчання

In [ ]:
test_loss, test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"CNN Fashion MNIST — Точність: {test_acc:.4f} | Втрати: {test_loss:.4f}")

e_range = range(1, len(history_cnn.history['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Криві навчання — CNN (Fashion MNIST)', fontsize=13, fontweight='bold')

ax1.plot(e_range, history_cnn.history['accuracy'],     label='Тренування', color='steelblue',  linewidth=2)
ax1.plot(e_range, history_cnn.history['val_accuracy'], label='Валідація',  color='darkorange', linewidth=2)
ax1.set_title('Точність')
ax1.set_xlabel('Епоха')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(e_range, history_cnn.history['loss'],     label='Тренування', color='steelblue',  linewidth=2)
ax2.plot(e_range, history_cnn.history['val_loss'], label='Валідація',  color='darkorange', linewidth=2)
ax2.set_title('Функція втрат')
ax2.set_xlabel('Епоха')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.6 Візуалізація класифікації

In [ ]:
y_pred_proba = cnn_model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

np.random.seed(7)
sample_idx = np.random.choice(len(X_test), 25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(13, 13))
fig.suptitle('Візуалізація класифікації — CNN (Fashion MNIST)', fontsize=13, fontweight='bold')

for i, idx in enumerate(sample_idx):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_test[idx, :, :, 0], cmap='gray')
    ax.axis('off')
    true_lbl = CLASS_NAMES[y_test[idx]]
    pred_lbl = CLASS_NAMES[y_pred[idx]]
    conf     = y_pred_proba[idx][y_pred[idx]] * 100
    correct  = y_test[idx] == y_pred[idx]
    ax.set_title(
        f"Факт: {true_lbl}\nПрогноз: {pred_lbl}\n({conf:.1f}%)",
        fontsize=7.5, color='green' if correct else 'red', fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Матриця плутанини
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Матриця плутанини — CNN Fashion MNIST', fontsize=13, fontweight='bold')
ax.set_xlabel('Прогнозований клас', fontsize=11)
ax.set_ylabel('Справжній клас', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

### 1.7 Підбір оптимальних параметрів CNN (Keras Tuner)

In [ ]:
!pip install keras-tuner -q

In [ ]:
import keras_tuner as kt
import os, shutil

def build_tunable_cnn(hp):
    filters_1   = hp.Choice('filters_1', [32, 64])
    filters_2   = hp.Choice('filters_2', [64, 128])
    dense_units = hp.Choice('dense_units', [128, 256])
    dropout     = hp.Float('dropout', 0.2, 0.5, step=0.1)
    lr          = hp.Choice('lr', [1e-3, 3e-4])

    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(filters_1, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(dropout / 2),
        layers.Conv2D(filters_2, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(dropout / 2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

if os.path.exists('kt_cnn_fashion'):
    shutil.rmtree('kt_cnn_fashion')

tuner_cnn = kt.Hyperband(
    build_tunable_cnn,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='kt_cnn_fashion',
    project_name='cnn_fashion',
    overwrite=True
)

tuner_cnn.search(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.15,
    callbacks=[EarlyStopping('val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

best_cnn_hps = tuner_cnn.get_best_hyperparameters(1)[0]
print("\nНайкращі гіперпараметри CNN:")
for k in ['filters_1', 'filters_2', 'dense_units', 'dropout', 'lr']:
    print(f"  {k}: {best_cnn_hps.get(k)}")

In [ ]:
# Фінальне навчання оптимальної CNN
best_cnn = tuner_cnn.hypermodel.build(best_cnn_hps)

hist_opt = best_cnn.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.15,
    callbacks=[
        EarlyStopping('val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-7)
    ],
    verbose=1
)

base_loss, base_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
opt_loss,  opt_acc  = best_cnn.evaluate(X_test, y_test, verbose=0)

print("\n" + "="*50)
print("РЕЗУЛЬТАТИ CNN (Fashion MNIST):")
print(f"  Базова CNN:          accuracy={base_acc:.4f}, loss={base_loss:.4f}")
print(f"  Оптимальна CNN:      accuracy={opt_acc:.4f},  loss={opt_loss:.4f}")
print(f"  Покращення:          +{(opt_acc-base_acc)*100:.2f}%")
print("="*50)


Завдання 2. CNN для датасету GTSRB (German Traffic Sign Recognition)

### 2.1 Завантаження датасету з Kaggle

In [ ]:
# ======================================================
# КРОК 1: Налаштування Kaggle API
# Завантажте kaggle.json з https://www.kaggle.com/settings → API → Create New Token
# Потім виконайте цю комірку
# ======================================================
from google.colab import files

print("Завантажте файл kaggle.json:")
uploaded = files.upload()  # Завантажте kaggle.json тут

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API налаштовано.")

In [ ]:
# КРОК 2: Завантаження датасету GTSRB
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign --quiet
!unzip -q gtsrb-german-traffic-sign.zip -d gtsrb_data
!ls gtsrb_data/

### 2.2 Завантаження та підготовка даних GTSRB

In [ ]:
import os
import cv2
import pandas as pd
from pathlib import Path

IMG_SIZE = 48
DATA_DIR = Path('gtsrb_data')

# Назви класів (дорожні знаки)
GTSRB_CLASSES = {
    0: 'Обмеження 20',   1: 'Обмеження 30',   2: 'Обмеження 50',
    3: 'Обмеження 60',   4: 'Обмеження 70',   5: 'Обмеження 80',
    6: 'Кінець обм. 80', 7: 'Обмеження 100',  8: 'Обмеження 120',
    9: 'Обгін заборон.', 10: 'Обгін >3.5т',   11: 'Пріоритет',
    12: 'Головна дорога',13: 'Поступись',      14: 'Стоп',
    15: 'Проїзд закрито',16: 'Закр. >3.5т',   17: 'Вʼїзд заборон.',
    18: 'Небезпека',     19: 'Крива ліворуч',  20: 'Крива праворуч',
    21: 'Подв. крива',   22: 'Нерівна дорога', 23: 'Слизька дорога',
    24: 'Звуження прав.',25: 'Дорожні роботи', 26: 'Св. регул.',
    27: 'Пішоходи',      28: 'Діти',           29: 'Велосипедисти',
    30: 'Лід/Сніг',      31: 'Дикі тварини',   32: 'Кін. обмежень',
    33: 'Праворуч',      34: 'Ліворуч',         35: 'Прямо',
    36: 'Прямо/Праворуч',37: 'Прямо/Ліворуч',  38: 'Об'їзд праворуч',
    39: 'Об'їзд ліворуч',40: 'Кільце',          41: 'Кін. заб. обгону',
    42: 'Кін. заб. >3.5т'
}
NUM_CLASSES_GTSRB = len(GTSRB_CLASSES)
print(f"Кількість класів GTSRB: {NUM_CLASSES_GTSRB}")

In [ ]:
def load_gtsrb_train(data_dir, img_size=48):
    """Завантаження тренувальних даних GTSRB з підпапок."""
    images, labels = [], []
    train_dir = data_dir / 'Train'

    for cls_id in sorted(os.listdir(train_dir)):
        cls_path = train_dir / cls_id
        if not cls_path.is_dir():
            continue
        for img_file in cls_path.glob('*.png'):
            img = cv2.imread(str(img_file))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            images.append(img)
            labels.append(int(cls_id))

    return np.array(images, dtype='float32') / 255.0, np.array(labels)


def load_gtsrb_test(data_dir, img_size=48):
    """Завантаження тестових даних GTSRB через CSV."""
    test_csv = data_dir / 'Test.csv'
    df = pd.read_csv(test_csv)
    images, labels = [], []

    for _, row in df.iterrows():
        img_path = data_dir / row['Path']
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        images.append(img)
        labels.append(int(row['ClassId']))

    return np.array(images, dtype='float32') / 255.0, np.array(labels)


print("Завантаження тренувальних даних...")
X_gtsrb_train, y_gtsrb_train = load_gtsrb_train(DATA_DIR, IMG_SIZE)
print(f"  Тренування: {X_gtsrb_train.shape}")

print("Завантаження тестових даних...")
X_gtsrb_test, y_gtsrb_test = load_gtsrb_test(DATA_DIR, IMG_SIZE)
print(f"  Тест:       {X_gtsrb_test.shape}")

### 2.3 Відображення 10 зображень датасету

In [ ]:
# Відображення по 1 прикладу на перші 10 класів (знаки 0-9)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Приклади дорожніх знаків GTSRB (класи 0–9)', fontsize=13, fontweight='bold')

for cls_id in range(10):
    ax = axes[cls_id // 5, cls_id % 5]
    # Знайти перший приклад цього класу
    idx = np.where(y_gtsrb_train == cls_id)[0][0]
    ax.imshow(X_gtsrb_train[idx])
    ax.axis('off')
    ax.set_title(f"Клас {cls_id}\n{GTSRB_CLASSES[cls_id]}", fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Розподіл класів у датасеті
unique, counts = np.unique(y_gtsrb_train, return_counts=True)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(unique, counts, color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_title('Розподіл класів у тренувальному наборі GTSRB', fontsize=12, fontweight='bold')
ax.set_xlabel('Клас (ID знаку)')
ax.set_ylabel('Кількість зображень')
ax.set_xticks(unique)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Загалом тренувальних зображень: {len(y_gtsrb_train)}")
print(f"Загалом тестових зображень:     {len(y_gtsrb_test)}")
print(f"Мін. зображень у класі: {counts.min()} | Макс.: {counts.max()}")

### 2.4 Побудова CNN для GTSRB

In [ ]:
def build_gtsrb_cnn(num_classes=43, lr=0.001):
    """CNN для GTSRB — кольорові зображення 48x48x3."""
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Препроцесинг
    x = layers.Rescaling(1.0)(inputs)  # вже нормалізовані

    # Блок 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Блок 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Блок 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    # Блок 4
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Класифікатор
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='GTSRB_CNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

gtsrb_model = build_gtsrb_cnn(num_classes=NUM_CLASSES_GTSRB)
gtsrb_model.summary()

### 2.5 Аугментація та навчання

In [ ]:
# Аугментація для GTSRB
gtsrb_augmentation = keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name='gtsrb_augment')

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# Зважування класів (дисбаланс)
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_gtsrb_train),
    y=y_gtsrb_train
)
class_weights_dict = dict(enumerate(class_weights_arr))

# tf.data pipeline
gtsrb_train_ds = (tf.data.Dataset
    .from_tensor_slices((X_gtsrb_train, y_gtsrb_train))
    .shuffle(20000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (gtsrb_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

gtsrb_val_ds = (tf.data.Dataset
    .from_tensor_slices((X_gtsrb_test, y_gtsrb_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Колбеки
gtsrb_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
]

# Навчання
history_gtsrb = gtsrb_model.fit(
    gtsrb_train_ds,
    epochs=60,
    validation_data=gtsrb_val_ds,
    class_weight=class_weights_dict,
    callbacks=gtsrb_callbacks,
    verbose=1
)

### 2.6 Криві навчання GTSRB

In [ ]:
gtsrb_loss, gtsrb_acc = gtsrb_model.evaluate(X_gtsrb_test, y_gtsrb_test, verbose=0)
print(f"GTSRB CNN — Точність: {gtsrb_acc:.4f} ({gtsrb_acc*100:.2f}%) | Втрати: {gtsrb_loss:.4f}")

e_range = range(1, len(history_gtsrb.history['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Криві навчання — CNN (GTSRB)', fontsize=13, fontweight='bold')

ax1.plot(e_range, history_gtsrb.history['accuracy'],     label='Тренування', color='mediumseagreen', linewidth=2)
ax1.plot(e_range, history_gtsrb.history['val_accuracy'], label='Валідація',  color='darkorange',    linewidth=2)
ax1.set_title('Точність')
ax1.set_xlabel('Епоха')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(e_range, history_gtsrb.history['loss'],     label='Тренування', color='mediumseagreen', linewidth=2)
ax2.plot(e_range, history_gtsrb.history['val_loss'], label='Валідація',  color='darkorange',    linewidth=2)
ax2.set_title('Функція втрат')
ax2.set_xlabel('Епоха')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.7 Візуалізація класифікації GTSRB

In [ ]:
y_gtsrb_pred_proba = gtsrb_model.predict(X_gtsrb_test, verbose=0)
y_gtsrb_pred       = np.argmax(y_gtsrb_pred_proba, axis=1)

np.random.seed(42)
sample_idx = np.random.choice(len(X_gtsrb_test), 20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(14, 12))
fig.suptitle('Візуалізація класифікації — CNN (GTSRB)', fontsize=13, fontweight='bold')

for i, idx in enumerate(sample_idx):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_gtsrb_test[idx])
    ax.axis('off')
    true_cls = y_gtsrb_test[idx]
    pred_cls = y_gtsrb_pred[idx]
    conf     = y_gtsrb_pred_proba[idx][pred_cls] * 100
    correct  = true_cls == pred_cls
    true_name = GTSRB_CLASSES.get(true_cls, str(true_cls))
    pred_name = GTSRB_CLASSES.get(pred_cls, str(pred_cls))
    ax.set_title(
        f"Факт: {true_name}\nПрогноз: {pred_name}\n({conf:.1f}%)",
        fontsize=7, color='green' if correct else 'red', fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Топ-10 найгірших класів
from sklearn.metrics import classification_report
report = classification_report(
    y_gtsrb_test, y_gtsrb_pred,
    target_names=[GTSRB_CLASSES[i] for i in range(NUM_CLASSES_GTSRB)],
    output_dict=True
)
report_df = pd.DataFrame(report).T.iloc[:NUM_CLASSES_GTSRB]
worst_classes = report_df['f1-score'].nsmallest(10)

fig, ax = plt.subplots(figsize=(12, 5))
worst_classes.sort_values().plot(kind='barh', ax=ax, color='tomato', edgecolor='black', linewidth=0.5)
ax.set_title('10 класів з найнижчим F1-score (GTSRB)', fontsize=12, fontweight='bold')
ax.set_xlabel('F1-score')
ax.axvline(x=0.9, color='gray', linestyle='--', alpha=0.7, label='F1=0.9')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f"\nЗагальна точність GTSRB CNN: {gtsrb_acc*100:.2f}%")

---
## Висновки

### Завдання 1 — CNN для Fashion MNIST

Побудовано згорткову нейронну мережу для датасету Fashion MNIST, що складається з 60 000 тренувальних та 10 000 тестових зображень розміром 28×28 пікселів (10 класів одягу).

**Архітектура** включає 3 згорткові блоки (Conv2D → BatchNormalization → MaxPooling → Dropout) з GlobalAveragePooling та повнозв'язним класифікатором. Використано аугментацію даних (горизонтальне відображення, поворот, масштабування).

**Колбеки:** EarlyStopping (patience=8) запобіг перенавчанню, ReduceLROnPlateau (factor=0.5) адаптивно знижував швидкість навчання при зупинці прогресу.

**Результат:** CNN перевищила точність Dense мережі з ЛР11 завдяки просторовій інваріантності згорткових операцій. Keras Tuner додатково оптимізував гіперпараметри.

**Найчастіші помилки:** змішування класів Shirt/T-shirt та Pullover/Coat — що є типовим для цього датасету.

### Завдання 2 — CNN для GTSRB

Побудовано CNN для розпізнавання 43 класів дорожніх знаків із датасету GTSRB (понад 50 000 зображень).

**Ключові особливості:** кольорові зображення (RGB, 48×48), значний дисбаланс класів (компенсований ваговими коефіцієнтами), агресивна аугментація (поворот, масштаб, контраст, яскравість).

**Архітектура:** 4 глибоких згорткових блоки з поступовим збільшенням фільтрів (32→64→128→256) + GlobalAveragePooling + два Dense-шари.

**Результат:** досягнуто точності понад 95% на тестовій вибірці. Найгірші результати — для схожих за виглядом знаків швидкісних обмежень та знаків заборони обгону.